### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [3]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model



In [4]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [5]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [6]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/pranav/Documents/AI projects/Churn Mode

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/50
167/167 [==============================] - 1s 1ms/step - loss: 0.4929 - accuracy: 0.7924
Epoch 2/50
167/167 [==============================] - 1s 1ms/step - loss: 0.5250 - accuracy: 0.7609
Epoch 2/50
167/167 [==============================] - 1s 2ms/step - loss: 0.5591 - accuracy: 0.7387
Epoch 2/50
167/167 [==============================] - 1s 1ms/step - loss: 0.4893 - accuracy: 0.7937
Epoch 2/50
167/167 [==============================] - 1s 1ms/step - loss: 0.4976 - accuracy: 0.7739
Epoch 2/50
167/167 [==============================] - 1s 2ms/step - loss: 0.6140 - accuracy: 0.6764
Epoch 2/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4506 - accuracy: 0.8027
Epoch 3/50
167/167 [==============================] - 1s 1ms/step - loss: 0.6599 - accuracy: 0.6121
Epoch 2/50
167/167 [==============================] - 0s 838us/step - loss: 0.4467 - accuracy: 0.8078
Epoch 3/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4217 - accuracy: 0.817

...

167/167 [==============================] - 0s 549us/step - loss: 0.3017 - accuracy: 0.8716
Epoch 99/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2864 - accuracy: 0.8825
Epoch 99/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2715 - accuracy: 0.8894
Epoch 36/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2647 - accuracy: 0.8892
Epoch 38/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2986 - accuracy: 0.8761
Epoch 100/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2984 - accuracy: 0.8793
Epoch 98/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2912 - accuracy: 0.8768
Epoch 40/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2616 - accuracy: 0.8890
Epoch 93/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2965 - accuracy: 0.8762
Epoch 42/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2984 - accurac

/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


167/167 [==============================] - 0s 1ms/step - loss: 0.2898 - accuracy: 0.8800
Epoch 43/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2601 - accuracy: 0.8877
Epoch 96/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2641 - accuracy: 0.8907
Epoch 40/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2568 - accuracy: 0.8871
Epoch 42/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2950 - accuracy: 0.8787
Epoch 45/100
167/167 [==============================] - 1s 1ms/step - loss: 0.4661 - accuracy: 0.7912
Epoch 2/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2904 - accuracy: 0.8791
Epoch 44/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2593 - accuracy: 0.8894
Epoch 97/100
167/167 [==============================] - 0s 1ms/step - loss: 0.2619 - accuracy: 0.8927
Epoch 41/100
167/167 [==============================] - 0s 1ms/step - loss: 0.4394 - accuracy: 0

/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


250/250 [==============================] - 0s 282us/step - loss: 0.5052 - accuracy: 0.7579
Epoch 2/100
250/250 [==============================] - 0s 255us/step - loss: 0.4179 - accuracy: 0.8166
Epoch 3/100
250/250 [==============================] - 0s 252us/step - loss: 0.3925 - accuracy: 0.8350
Epoch 4/100
250/250 [==============================] - 0s 251us/step - loss: 0.3730 - accuracy: 0.8459
Epoch 5/100
250/250 [==============================] - 0s 456us/step - loss: 0.3609 - accuracy: 0.8514
Epoch 6/100
250/250 [==============================] - 0s 252us/step - loss: 0.3535 - accuracy: 0.8565
Epoch 7/100
250/250 [==============================] - 0s 250us/step - loss: 0.3494 - accuracy: 0.8562
Epoch 8/100
250/250 [==============================] - 0s 247us/step - loss: 0.3468 - accuracy: 0.8575
Epoch 9/100
250/250 [==============================] - 0s 249us/step - loss: 0.3442 - accuracy: 0.8572
Epoch 10/100
250/250 [==============================] - 0s 247us/step - loss: 0.3422 

In [8]:
grid.best_estimator_.model_.save("best_ann_model.h5")

/Users/pranav/Documents/AI projects/Churn Modelling/annclassification/venv/lib/python3.9/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [9]:
from tensorflow.keras.models import load_model
m = load_model("best_ann_model.h5")
print("Loaded model summary:")
m.summary()

Loaded model summary:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 897 (3.50 KB)
Trainable params: 897 (3.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
